# Cross-Modal Fusion Analysis — Dopaminergic Voice Project

| Field | Value |
|-------|-------|
| **PI** | Peter Zhou |
| **Stream A Lead** | Lily Wu |
| **Stream B Lead** | Dora Xiang |
| **Design** | Cross-modal comparison (independent cohorts) |
| **Target Output** | Ablation table, ROC comparisons, SHAP plots |

## Architecture

```
Stream A (DAIC-WOZ)          Stream B (ds000030)
┌──────────────────┐         ┌──────────────────┐
│ Acoustic Prosody │         │  NAcc BOLD fMRI  │
│ f0, energy, etc  │         │ bilateral mean/  │
│  189 subjects    │         │  std, asymmetry  │
│ Label: PHQ-8     │         │  ~272 subjects   │
│ (depression)     │         │ Label: BDI/MASQ  │
└────────┬─────────┘         │ (anhedonia)      │
         │                   └────────┬─────────┘
         v                            v
   Classifier A                Classifier B
   (LogReg, RF)                (LogReg, RF)
         │                            │
         └──────────┬─────────────────┘
                   v
        Cross-Modal Comparison
        • Ablation table
        • ROC overlay
        • SHAP feature importance
        • Effect size comparison
```

> ⚠️ **These are independent cohorts** — DAIC-WOZ and ds000030 have different subjects. We compare classification performance *across modalities*, we do NOT merge subjects.

---
## Cell 1: Dependencies

In [ ]:
# ============================================================
# Cell 1: Install + import
# ============================================================
!pip install -q pandas scikit-learn matplotlib seaborn shap

import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import (
    StratifiedKFold, cross_val_score, cross_val_predict
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score,
    precision_recall_curve, average_precision_score
)
from collections import OrderedDict

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 120

print('✅ Dependencies loaded')

---
## Cell 2: Mount Drive & Configure Paths

In [ ]:
# ============================================================
# Cell 2: Configuration
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/NSG/Fusion'
os.makedirs(DRIVE_BASE, exist_ok=True)

# ---- INPUT PATHS ---- #
# Update these to your actual CSV locations on Drive
STREAM_A_CSV = '/content/drive/MyDrive/NSG/Stream_A/stream_a_features.csv'
STREAM_B_CSV = '/content/drive/MyDrive/NSG/Stream_B/stream_b_features.csv'

# ---- OUTPUT DIR ---- #
RESULTS_DIR = DRIVE_BASE

# ---- ANALYSIS CONFIG ---- #
CV_FOLDS = 5
RANDOM_STATE = 42
MIN_SAMPLES = 20  # Minimum to run classification

print(f'Stream A CSV: {STREAM_A_CSV}')
print(f'Stream B CSV: {STREAM_B_CSV}')
print(f'Results → {RESULTS_DIR}')

---
## Cell 3: Load Stream Data

Each stream is loaded independently with its own cohort-specific label.

In [ ]:
# ============================================================
# Cell 3: Load stream CSVs
# ============================================================

streams = OrderedDict()

# --- Stream A: DAIC-WOZ Acoustic Features --- #
if os.path.exists(STREAM_A_CSV):
    df_a = pd.read_csv(STREAM_A_CSV)
    
    # Identify feature columns (exclude ID and label columns)
    id_cols_a = [c for c in df_a.columns if 'participant' in c.lower() or 'id' in c.lower()]
    label_col_a = [c for c in df_a.columns if c.lower() in 
                   ['label', 'phq8_binary', 'phq_binary', 'depression', 'phq8_score']]
    
    # Auto-detect or set label
    if label_col_a:
        df_a['label'] = df_a[label_col_a[0]]
    else:
        # For development: create binary label from PHQ-8 if available
        phq_cols = [c for c in df_a.columns if 'phq' in c.lower()]
        if phq_cols:
            df_a['label'] = (df_a[phq_cols[0]] >= 10).astype(int)
        else:
            print('  ⚠ No depression label found in Stream A — using synthetic labels')
            np.random.seed(RANDOM_STATE)
            df_a['label'] = np.random.randint(0, 2, size=len(df_a))
    
    feature_cols_a = [c for c in df_a.columns 
                      if c not in id_cols_a + label_col_a + ['label']]
    
    streams['Stream A\n(Acoustic)'] = {
        'df': df_a,
        'features': feature_cols_a,
        'label_col': 'label',
        'cohort': 'DAIC-WOZ',
        'target': 'Depression (PHQ-8 ≥ 10)',
        'color': '#2196F3'
    }
    print(f'✅ Stream A: {df_a.shape[0]} subjects × {len(feature_cols_a)} features')
    print(f'   Label distribution: {dict(df_a["label"].value_counts())}')
else:
    print(f'⚠️  Stream A not found at {STREAM_A_CSV}')

# --- Stream B: ds000030 fMRI Features --- #
if os.path.exists(STREAM_B_CSV):
    df_b = pd.read_csv(STREAM_B_CSV)
    
    id_cols_b = [c for c in df_b.columns if 'subject' in c.lower() or 'participant' in c.lower()]
    
    # ds000030 label: binarize BDI-II or MASQ anhedonia
    label_col_b = [c for c in df_b.columns if any(kw in c.lower() 
                   for kw in ['bdi', 'masq', 'label', 'depression', 'anhedon'])]
    
    if label_col_b:
        score_col = label_col_b[0]
        if df_b[score_col].nunique() > 2:
            # Continuous → binary: median split
            threshold = df_b[score_col].median()
            df_b['label'] = (df_b[score_col] >= threshold).astype(int)
            print(f'   Binarized {score_col} at median ({threshold:.1f})')
        else:
            df_b['label'] = df_b[score_col]
    else:
        print('  ⚠ No clinical label found in Stream B — using synthetic labels')
        np.random.seed(RANDOM_STATE + 1)
        df_b['label'] = np.random.randint(0, 2, size=len(df_b))
    
    # NAcc BOLD features
    feature_cols_b = [c for c in df_b.columns if c.startswith('nacc_') or 
                      c in ['n_volumes']]
    if not feature_cols_b:
        feature_cols_b = [c for c in df_b.columns 
                          if c not in id_cols_b + label_col_b + ['label', 'bold_file']]
    
    streams['Stream B\n(fMRI)'] = {
        'df': df_b,
        'features': feature_cols_b,
        'label_col': 'label',
        'cohort': 'ds000030 (UCLA CNP)',
        'target': 'Anhedonia (BDI/MASQ median split)',
        'color': '#F44336'
    }
    print(f'✅ Stream B: {df_b.shape[0]} subjects × {len(feature_cols_b)} features')
    print(f'   Label distribution: {dict(df_b["label"].value_counts())}')
else:
    print(f'⚠️  Stream B not found at {STREAM_B_CSV}')

print(f'\n📊 {len(streams)} stream(s) loaded for analysis')

---
## Cell 4: Per-Stream Classification (Independent Cohorts)

Each stream runs its own `StratifiedKFold` cross-validation. We compare F1, AUC, and precision-recall **across modalities** (not across merged participants).

In [ ]:
# ============================================================
# Cell 4: Run independent classifiers per stream
# ============================================================

MODEL_SUITE = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=5, random_state=RANDOM_STATE,
        class_weight='balanced'
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, max_depth=3, random_state=RANDOM_STATE
    ),
}

all_results = {}  # {stream_name: {model_name: {metrics}}}
roc_data = {}     # For overlay plot
best_models = {}  # {stream_name: fitted best model}

for stream_name, stream_info in streams.items():
    df = stream_info['df']
    feat_cols = stream_info['features']
    
    X = df[feat_cols].values
    y = df[stream_info['label_col']].values
    
    # Drop NaN rows
    mask = ~np.isnan(X).any(axis=1) & ~np.isnan(y)
    X, y = X[mask], y[mask]
    
    print(f'\n{"="*60}')
    print(f'{stream_name.replace(chr(10), " ")} — {stream_info["cohort"]}')
    print(f'  Subjects: {len(X)} | Features: {X.shape[1]}')
    print(f'  Target: {stream_info["target"]}')
    print(f'  Class balance: {dict(zip(*np.unique(y, return_counts=True)))}')
    print(f'{"="*60}')
    
    if len(X) < MIN_SAMPLES:
        print(f'  ⚠ Skipping — need ≥{MIN_SAMPLES} samples, have {len(X)}')
        print(f'  Run Stream A/B notebooks first to generate feature CSVs.')
        continue
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    
    stream_results = {}
    
    for model_name, clf in MODEL_SUITE.items():
        # Cross-validated predictions (held-out — no data leakage)
        y_pred = cross_val_predict(clf, X_scaled, y, cv=cv)
        
        f1_scores = cross_val_score(clf, X_scaled, y, cv=cv, scoring='f1')
        auc_scores = cross_val_score(clf, X_scaled, y, cv=cv, scoring='roc_auc')
        
        stream_results[model_name] = {
            'f1_mean': f1_scores.mean(),
            'f1_std': f1_scores.std(),
            'auc_mean': auc_scores.mean(),
            'auc_std': auc_scores.std(),
        }
        
        print(f'  {model_name:25s}  F1={f1_scores.mean():.3f}±{f1_scores.std():.3f}  '
              f'AUC={auc_scores.mean():.3f}±{auc_scores.std():.3f}')
    
    all_results[stream_name] = stream_results
    
    # Fit best model for ROC + SHAP
    best_name = max(stream_results, key=lambda k: stream_results[k]['auc_mean'])
    best_clf = MODEL_SUITE[best_name].__class__(**MODEL_SUITE[best_name].get_params())
    best_clf.fit(X_scaled, y)
    best_models[stream_name] = (best_clf, scaler, feat_cols, X_scaled, y)
    
    # Cross-validated ROC (held-out probabilities)
    y_proba = cross_val_predict(
        MODEL_SUITE[best_name].__class__(**MODEL_SUITE[best_name].get_params()),
        X_scaled, y, cv=cv, method='predict_proba'
    )[:, 1]
    roc_data[stream_name] = (y, y_proba, stream_info['color'])
    
    print(f'  → Best: {best_name}')

print(f'\n✅ Classification complete for {len(all_results)} stream(s)')

---
## Cell 5: Ablation Table

Comparison of classification performance across modalities — the core finding of the paper.

In [ ]:
# ============================================================
# Cell 5: Ablation table
# ============================================================

if all_results:
    rows = []
    for stream_name, models in all_results.items():
        stream_label = stream_name.replace('\n', ' ')
        cohort = streams[stream_name]['cohort']
        n = len(streams[stream_name]['df'])
        
        for model_name, metrics in models.items():
            rows.append({
                'Modality': stream_label,
                'Cohort': cohort,
                'N': n,
                'Model': model_name,
                'F1 (CV)': f"{metrics['f1_mean']:.3f} ± {metrics['f1_std']:.3f}",
                'AUC (CV)': f"{metrics['auc_mean']:.3f} ± {metrics['auc_std']:.3f}",
            })
    
    df_ablation = pd.DataFrame(rows)
    
    # Save
    ablation_path = os.path.join(RESULTS_DIR, 'ablation_table.csv')
    df_ablation.to_csv(ablation_path, index=False)
    print('📋 Ablation Table\n')
    display(df_ablation)
    print(f'\nSaved to {ablation_path}')
else:
    print('No classification results — run Cell 4 with sufficient data.')

---
## Cell 6: ROC Curves — Cross-Modal Overlay

Overlays the held-out ROC curves from each stream on one plot to visually compare discriminability.

In [ ]:
# ============================================================
# Cell 6: ROC overlay
# ============================================================

if roc_data:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # --- Panel A: ROC Curves --- #
    ax = axes[0]
    for stream_name, (y_true, y_prob, color) in roc_data.items():
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc = roc_auc_score(y_true, y_prob)
        label = stream_name.replace('\n', ' ')
        ax.plot(fpr, tpr, label=f'{label} (AUC={auc:.3f})',
                color=color, linewidth=2)
    
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, linewidth=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('A. ROC Curves (Held-Out CV)', fontweight='bold')
    ax.legend(loc='lower right', frameon=True)
    ax.set_aspect('equal')
    
    # --- Panel B: Precision-Recall Curves --- #
    ax = axes[1]
    for stream_name, (y_true, y_prob, color) in roc_data.items():
        prec, rec, _ = precision_recall_curve(y_true, y_prob)
        ap = average_precision_score(y_true, y_prob)
        label = stream_name.replace('\n', ' ')
        ax.plot(rec, prec, label=f'{label} (AP={ap:.3f})',
                color=color, linewidth=2)
    
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title('B. Precision-Recall Curves (Held-Out CV)', fontweight='bold')
    ax.legend(loc='upper right', frameon=True)
    
    plt.tight_layout()
    fig_path = os.path.join(RESULTS_DIR, 'roc_pr_comparison.png')
    plt.savefig(fig_path, dpi=200, bbox_inches='tight')
    print(f'📊 Saved to {fig_path}')
    plt.show()
else:
    print('No ROC data — classification did not run.')

---
## Cell 7: SHAP Feature Importance (Per Stream)

SHAP (SHapley Additive exPlanations) shows which features drive predictions. This is critical for interpreting *why* each modality succeeds or fails.

In [ ]:
# ============================================================
# Cell 7: SHAP plots per stream
# ============================================================

for stream_name, (clf, scaler, feat_names, X_scaled, y) in best_models.items():
    label = stream_name.replace('\n', ' ')
    print(f'\n{"="*50}')
    print(f'SHAP Analysis — {label}')
    print(f'{"="*50}')
    
    # Use TreeExplainer for tree models, KernelExplainer for linear
    if hasattr(clf, 'estimators_'):
        explainer = shap.TreeExplainer(clf)
        shap_values = explainer.shap_values(X_scaled)
        # For binary: take class-1 SHAP values
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
    else:
        # Linear model — use masker-based explainer
        explainer = shap.LinearExplainer(clf, X_scaled)
        shap_values = explainer.shap_values(X_scaled)
    
    # Beeswarm plot
    fig, ax = plt.subplots(figsize=(10, max(4, len(feat_names) * 0.4)))
    shap.summary_plot(
        shap_values, X_scaled,
        feature_names=feat_names,
        show=False, max_display=15
    )
    plt.title(f'{label} — SHAP Feature Importance', fontweight='bold')
    plt.tight_layout()
    shap_path = os.path.join(RESULTS_DIR, f'shap_{label.replace(" ", "_").lower()}.png')
    plt.savefig(shap_path, dpi=200, bbox_inches='tight')
    print(f'📊 Saved to {shap_path}')
    plt.show()
    
    # Top features table
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    top_df = pd.DataFrame({
        'Feature': feat_names,
        'Mean |SHAP|': mean_abs_shap
    }).sort_values('Mean |SHAP|', ascending=False).head(10)
    print(f'\nTop features:')
    display(top_df)

---
## Cell 8: Confusion Matrices

In [ ]:
# ============================================================
# Cell 8: Confusion matrices (held-out predictions)
# ============================================================

if roc_data:
    n_streams = len(roc_data)
    fig, axes = plt.subplots(1, n_streams, figsize=(6 * n_streams, 5))
    if n_streams == 1:
        axes = [axes]
    
    for ax, (stream_name, (y_true, y_prob, color)) in zip(axes, roc_data.items()):
        y_pred = (y_prob >= 0.5).astype(int)
        cm = confusion_matrix(y_true, y_pred)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Negative', 'Positive'],
                    yticklabels=['Negative', 'Positive'])
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        label = stream_name.replace('\n', ' ')
        ax.set_title(f'{label}', fontweight='bold')
    
    plt.suptitle('Confusion Matrices (Held-Out CV Predictions)',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    cm_path = os.path.join(RESULTS_DIR, 'confusion_matrices.png')
    plt.savefig(cm_path, dpi=200, bbox_inches='tight')
    print(f'📊 Saved to {cm_path}')
    plt.show()

---
## Cell 9: Summary Report

In [ ]:
# ============================================================
# Cell 9: Export summary
# ============================================================

print('\n' + '=' * 60)
print('CROSS-MODAL FUSION ANALYSIS — SUMMARY REPORT')
print('=' * 60)

for stream_name, stream_info in streams.items():
    label = stream_name.replace('\n', ' ')
    n = len(stream_info['df'])
    n_feat = len(stream_info['features'])
    cohort = stream_info['cohort']
    target = stream_info['target']
    
    print(f'\n--- {label} ---')
    print(f'  Cohort:    {cohort}')
    print(f'  Subjects:  {n}')
    print(f'  Features:  {n_feat}')
    print(f'  Target:    {target}')
    
    if stream_name in all_results:
        best_name = max(all_results[stream_name],
                        key=lambda k: all_results[stream_name][k]['auc_mean'])
        m = all_results[stream_name][best_name]
        print(f'  Best Model: {best_name}')
        print(f'  F1 (CV):   {m["f1_mean"]:.3f} ± {m["f1_std"]:.3f}')
        print(f'  AUC (CV):  {m["auc_mean"]:.3f} ± {m["auc_std"]:.3f}')
    else:
        print(f'  ⚠ Classification not run (< {MIN_SAMPLES} samples)')

print(f'\n--- Output Files ---')
for f in sorted(os.listdir(RESULTS_DIR)):
    if not f.startswith('.'):
        fpath = os.path.join(RESULTS_DIR, f)
        size = os.path.getsize(fpath)
        print(f'  • {f} ({size:,} bytes)')

print('\n🎯 Key Question: Does acoustic prosody (Stream A) or')
print('   ventral striatal BOLD (Stream B) better discriminate')
print('   clinical depression / anhedonia?')
print('\n   → Compare AUC values across modalities above.')
print('\n✅ Analysis complete!')